<a href="https://colab.research.google.com/github/dhairyasheel9364/IITB-EdTech-Internship-2025/blob/main/kidneymodelfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
from google.colab import files

print("👇 UPLOAD YOUR kaggle.json FILE NOW 👇")
uploaded = files.upload()

# Secure the key
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download your custom, pre-shrunk dataset
print("\nDownloading your private dataset...")
!kaggle datasets download -d dhairyasheelwangdare/neurorenal-kidney-224

# Unzip it
print("Unzipping...")
!unzip -q neurorenal-kidney-224.zip -d kidney_data
print("✅ Dataset successfully downloaded and unzipped!")

👇 UPLOAD YOUR kaggle.json FILE NOW 👇


Saving kaggle.json to kaggle.json

Dataset URL: https://www.kaggle.com/datasets/dhairyasheelwangdare/neurorenal-kidney-224
License(s): unknown
neurorenal-kidney-224.zip: Skipping, found more recently modified local copy (use --force to force download)
Unzipping...
replace kidney_data/Kidney_Data_224_Ready/Cyst/Cyst- (1).jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
✅ Dataset successfully downloaded and unzipped!


In [ ]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from google.colab import files

# --- 1. THE "STRICT TEACHER" GENERATOR (Data Augmentation) ---
# This forces the AI to learn shapes, not just memorize pixels.
datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,       # Rotate images slightly
    width_shift_range=0.1,   # Shift left/right
    height_shift_range=0.1,  # Shift up/down
    shear_range=0.1,         # Distort slightly
    zoom_range=0.1,          # Zoom in/out
    horizontal_flip=True,    # Flip left-to-right
    fill_mode='nearest',
    validation_split=0.2     # 20% for testing
)

# NOTE: Ensure this path matches where your data was unzipped!
KIDNEY_DIR = '/content/kidney_data/Kidney_Data_224_Ready'

print("🔄 Loading Augmented Training Data...")
train_gen = datagen.flow_from_directory(
    KIDNEY_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

print("🔄 Loading Validation Data...")
val_gen = datagen.flow_from_directory(
    KIDNEY_DIR,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# --- 2. BUILD THE PROTECTED MODEL ---
# We keep the base FROZEN (weights='imagenet' and trainable=False)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'), # More neurons to capture medical details
    Dropout(0.5),                  # Stronger dropout to kill overfitting
    Dense(4, activation='softmax')
])

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# --- 3. THE MASTER RUN (15 EPOCHS) ---
print("\n🚀 Starting the Master Run. This will be slower but much smarter...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15
)

# --- 4. SAVE AND DOWNLOAD ---
weights_name = 'kidney_mobilenetv2_master.weights.h5'
model.save_weights(weights_name)
files.download(weights_name)

print(f"\n✅ Training Complete! Downloaded: {weights_name}")

🔄 Loading Augmented Training Data...
Found 9959 images belonging to 4 classes.
🔄 Loading Validation Data...
Found 2487 images belonging to 4 classes.

🚀 Starting the Master Run. This will be slower but much smarter...
Epoch 1/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 197s 572ms/step - accuracy: 0.7805 - loss: 0.5791 - val_accuracy: 0.6630 - val_loss: 0.8828
Epoch 2/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 148s 474ms/step - accuracy: 0.8888 - loss: 0.3038 - val_accuracy: 0.7330 - val_loss: 0.6796
Epoch 3/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 148s 476ms/step - accuracy: 0.9160 - loss: 0.2325 - val_accuracy: 0.6574 - val_loss: 1.0194
Epoch 4/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 147s 473ms/step - accuracy: 0.9319 - loss: 0.1880 - val_accuracy: 0.7169 - val_loss: 0.8031
Epoch 5/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 149s 478ms/step - accuracy: 0.9350 - loss: 0.1704 - val_accuracy: 0.7185 - val_loss: 0.7769
Epoch 6/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 149s 478ms/step - accuracy: 0.9516 - loss: 0.1334 - val_accuracy: 0.7053 - val_los

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Training Complete! Downloaded: kidney_mobilenetv2_master.weights.h5


In [ ]:
from sklearn.utils import class_weight
import numpy as np

# 1. Calculate weights to balance the 'Cyst' bias
# This tells the AI to treat 'Stone' images as more important
class_weights = {
    0: 1.0,  # Cyst
    1: 1.0,  # Normal
    2: 3.0,  # Stone (Give this 3x more importance!)
    3: 1.5   # Tumor
}

print("🚀 Restarting Training with Class Weights...")

# 2. Run the same Master Run but with the 'class_weight' parameter
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    class_weight=class_weights # <--- THE MAGIC FIX
)

# 3. Save as the Final V3
weights_name = 'kidney_mobilenetv2_weighted.weights.h5'
model.save_weights(weights_name)
files.download(weights_name)

🚀 Restarting Training with Class Weights...
Epoch 1/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 160s 492ms/step - accuracy: 0.9433 - loss: 0.2075 - val_accuracy: 0.6345 - val_loss: 1.2343
Epoch 2/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 148s 475ms/step - accuracy: 0.9591 - loss: 0.1647 - val_accuracy: 0.7041 - val_loss: 1.0250
Epoch 3/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 149s 479ms/step - accuracy: 0.9614 - loss: 0.1441 - val_accuracy: 0.6220 - val_loss: 1.7097
Epoch 4/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 148s 476ms/step - accuracy: 0.9665 - loss: 0.1255 - val_accuracy: 0.5927 - val_loss: 1.7914
Epoch 5/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 149s 479ms/step - accuracy: 0.9689 - loss: 0.1228 - val_accuracy: 0.7065 - val_loss: 1.5834
Epoch 6/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 148s 475ms/step - accuracy: 0.9695 - loss: 0.1106 - val_accuracy: 0.6803 - val_loss: 1.3180
Epoch 7/15
312/312 ━━━━━━━━━━━━━━━━━━━━ 147s 472ms/step - accuracy: 0.9722 - loss: 0.1104 - val_accuracy: 0.6715 - val_loss: 1.3953
Epoch 8/15
312/312 ━━━━━━━━━━━━━

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>